In [ ]:
# Build the panchayat read model.
#
# The schema, transformations, reset order and validation live in src/database
# so they can be reviewed, imported and tested. This notebook only drives them.
# Equivalent command line:
#
#     uv run python scripts/build_panchayat_db.py build

from database import config
from database.build import build

print("database:", config.DB_PATH)
print("planning:", config.PLANNING_CSV)


In [ ]:
# Rebuild and publish. Atomic: the target is replaced only after every table
# has loaded and validation has passed, so a failure leaves the previous
# database untouched.
counts = build()

for table, n in counts.items():
    print(f"{table:<30} {n:>8,}")


In [ ]:
# Validation gate. Expected counts come from src/database/manifest.yaml.
from database.validate import validate_database

for check in validate_database():
    print(f"{'PASS' if check.passed else 'FAIL'}  {check.name:<55} {check.detail}")


In [ ]:
# Anything the build could not load, and why. Empty is the healthy case.
import duckdb

con = duckdb.connect(str(config.DB_PATH), read_only=True)
try:
    display(con.execute("""
        SELECT table_name, reason, key_column, sum(row_count) AS rows
        FROM quarantine
        GROUP BY 1, 2, 3
        ORDER BY rows DESC
    """).df())
finally:
    con.close()
